# Table A — Step 2: ASpanFormer Geometry Filter (Cached)

Runs ASpanFormer on the SSCD and CLIP retrieval manifests from Step 1.
Uses reconstructed DINO `aspan_all_manifest` files as a failure cache:
pairs already confirmed as `aspan_pass=False` in the DINO run are copied directly
into the output manifest without re-running ASpanFormer.

**Why only cache failures?** Sidecar filenames embed the `candidate_id` (see
`geometry_filter.py:204-207`). SSCD/CLIP candidates have different `candidate_id`
values from DINO (different retrieval rank encoding), so old sidecars cannot be
reused. Passed pairs always get a fresh ASpanFormer run to generate new sidecars.

---

**Before running this notebook**, upload to Drive at `[DRIVE_ROOT]/scripts/`:
```
scripts/
    geometry_filter.py
    geometry_filter_cached.py
    ml-aspanformer/               ← the full repo directory (same one used by geometry_filter.py)
    _local/
        aspan_all_manifest_shard1_reconstructed.jsonl
        aspan_all_manifest_shard2_reconstructed.jsonl
```

**Inputs** (produced by `retrieval_ablation_colab.ipynb`):
```
ablation_outputs/retrieval/
    sscd_retrieval_manifest.jsonl
    clip_retrieval_manifest.jsonl
```

**Outputs** (written to Drive at the end):
```
ablation_outputs/geometry_filter/
    sscd_shard1/
        aspan_all_manifest.jsonl
        vggt_candidates_manifest.jsonl
        aspan_sidecars/
    sscd_shard2/  (same structure)
    clip_shard1/  (same structure)
    clip_shard2/  (same structure)
```

In [ ]:
# Step 0 — Verify GPU and pin einops for ASpanFormer compatibility
# einops>=0.6 removed _torch_specific which ml-aspanformer depends on
!pip install einops==0.3.0 -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — switch runtime to GPU before continuing.")

In [ ]:
# Step 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2 — Configure paths
# ── EDIT THESE ──────────────────────────────────────────────────────────────
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/ImageSimilarity')
SCRIPTS_DIR  = DRIVE_ROOT / 'scripts'
ASPANPATH    = '/content/ml-aspanformer'
WEIGHTS_PATH = DRIVE_ROOT / 'weights' / 'outdoor.ckpt'
CONFIG_PATH  = '/content/ml-aspanformer/configs/aspan/outdoor/aspan_test.py'

# Image corpus lives on Drive as zip files (same zips used by retrieval_ablation_colab.ipynb)
SOURCE_ZIP = DRIVE_ROOT / 'workingsourcecrops.zip'
TARGET_ZIP = DRIVE_ROOT / 'working_targetcrops.zip'

# After extraction these match the paths baked into the SSCD/CLIP retrieval manifests
LOCAL_SOURCE_DIR = Path('./source')
LOCAL_TARGET_DIR = Path('./target')

# Retrieval manifests from Step 1 Colab
SSCD_RETRIEVAL = DRIVE_ROOT / 'ablation_outputs' / 'retrieval' / 'sscd_retrieval_manifest.jsonl'
CLIP_RETRIEVAL = DRIVE_ROOT / 'ablation_outputs' / 'retrieval' / 'clip_retrieval_manifest.jsonl'

DRIVE_OUT   = DRIVE_ROOT / 'ablation_outputs' / 'geometry_filter'
BREAKPOINT  = 50    # keypoint threshold (must match original pipeline)
# ────────────────────────────────────────────────────────────────────────────

import shutil, sys, json, os

LOCAL_WORK = Path('/content/geometry_filter_ablation')
(LOCAL_WORK / '_local').mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

# Cache manifests live in _local/ after being copied from Drive
SHARD1_CACHE = LOCAL_WORK / '_local' / 'aspan_all_manifest_shard1_reconstructed.jsonl'
SHARD2_CACHE = LOCAL_WORK / '_local' / 'aspan_all_manifest_shard2_reconstructed.jsonl'

print(f"ASpanFormer path : {ASPANPATH}")
print(f"Weights          : {WEIGHTS_PATH}  (exists: {Path(str(WEIGHTS_PATH)).exists()})")
print(f"Source zip       : {SOURCE_ZIP}  (exists: {SOURCE_ZIP.exists()})")
print(f"Target zip       : {TARGET_ZIP}  (exists: {TARGET_ZIP.exists()})")
print(f"SSCD manifest    : {SSCD_RETRIEVAL}  (exists: {SSCD_RETRIEVAL.exists()})")
print(f"CLIP manifest    : {CLIP_RETRIEVAL}  (exists: {CLIP_RETRIEVAL.exists()})")

In [ ]:
# Step 3 — Copy scripts and cache manifests from Drive to Colab local filesystem
#
# Drive upload structure expected under scripts/:
#   scripts/
#       geometry_filter.py
#       geometry_filter_cached.py          ← sits alongside geometry_filter.py
#       _local/
#           aspan_all_manifest_shard1_reconstructed.jsonl
#           aspan_all_manifest_shard2_reconstructed.jsonl

to_copy = [
    # Scripts → LOCAL_WORK root (alongside geometry_filter.py)
    (SCRIPTS_DIR / 'geometry_filter.py',
     LOCAL_WORK / 'geometry_filter.py'),
    (SCRIPTS_DIR / 'geometry_filter_cached.py',
     LOCAL_WORK / 'geometry_filter_cached.py'),
    # Cache manifests → LOCAL_WORK/_local/
    (SCRIPTS_DIR / '_local' / 'aspan_all_manifest_shard1_reconstructed.jsonl',
     SHARD1_CACHE),
    (SCRIPTS_DIR / '_local' / 'aspan_all_manifest_shard2_reconstructed.jsonl',
     SHARD2_CACHE),
]

for src, dst in to_copy:
    src = Path(str(src))
    dst = Path(str(dst))
    if not src.exists():
        raise FileNotFoundError(
            f"Missing on Drive: {src}\nUpload it to {src.parent} and re-run."
        )
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(src), str(dst))
    size_mb = src.stat().st_size / 1e6
    print(f"Copied: {src.name}  ({size_mb:.1f} MB)")

sys.path.insert(0, str(LOCAL_WORK))
print("\nAll scripts and cache manifests ready.")
print(f"  Shard 1 cache: {SHARD1_CACHE}  (exists: {SHARD1_CACHE.exists()})")
print(f"  Shard 2 cache: {SHARD2_CACHE}  (exists: {SHARD2_CACHE.exists()})")

In [ ]:
# Step 3a — Copy ml-aspanformer from Drive
# Upload ml-aspanformer/ to Drive at scripts/ml-aspanformer/ (same place as the other scripts).
# This mirrors how the existing geometry_filter pipeline sets up ASpanFormer on Colab.

ASPAN_SRC = SCRIPTS_DIR / 'ml-aspanformer'

if not Path(ASPANPATH).exists():
    if not ASPAN_SRC.exists():
        raise FileNotFoundError(
            f"ml-aspanformer not found on Drive at {ASPAN_SRC}\n"
            "Upload the ml-aspanformer/ directory there and re-run."
        )
    print(f"Copying ml-aspanformer from Drive → {ASPANPATH} ...")
    shutil.copytree(str(ASPAN_SRC), ASPANPATH)
    print("Done.")
else:
    print(f"ml-aspanformer already present at {ASPANPATH}")

In [ ]:
# Step 3b — Extract and flatten image corpus from Drive zips
# Mirrors the setup in retrieval_ablation_colab.ipynb.
# Idempotent: skips extraction if directories already have files.

import time

def _extract_and_flatten(zip_src: Path, local_name: str) -> int:
    """Copy zip from Drive, extract, flatten subdirectories to top level."""
    dst_dir = Path(f'./{local_name}')
    if dst_dir.exists() and any(dst_dir.iterdir()):
        n = sum(1 for p in dst_dir.iterdir() if p.is_file())
        print(f"  {local_name}/: already present ({n:,} files)")
        return n

    local_zip = Path(f'./{local_name}.zip')
    print(f"  Copying {zip_src.name} from Drive ...")
    shutil.copy(str(zip_src), str(local_zip))

    print(f"  Unzipping → ./{local_name}/ ...")
    t0 = time.time()
    os.system(f'unzip -q {local_zip} -d {local_name}')
    print(f"  Unzipped in {time.time() - t0:.0f}s")

    # Flatten: move all files from subdirectories up to dst_dir
    for root, dirs, files in os.walk(str(dst_dir)):
        if root == str(dst_dir):
            continue
        for fname in files:
            src_path = os.path.join(root, fname)
            dst_path = os.path.join(str(dst_dir), fname)
            if os.path.exists(dst_path):
                base, ext = os.path.splitext(fname)
                dst_path = os.path.join(str(dst_dir), f"{base}_{root.replace('/', '_')}{ext}")
            shutil.move(src_path, dst_path)

    # Remove now-empty subdirectories
    for root, dirs, files in os.walk(str(dst_dir), topdown=False):
        if root == str(dst_dir):
            continue
        try:
            os.rmdir(root)
        except OSError:
            pass

    local_zip.unlink(missing_ok=True)
    n = sum(1 for p in dst_dir.iterdir() if p.is_file())
    print(f"  {local_name}/: {n:,} files ready")
    return n

n_src = _extract_and_flatten(SOURCE_ZIP, 'source')
n_tgt = _extract_and_flatten(TARGET_ZIP, 'target')
print(f"\nImages ready: {n_src:,} sources, {n_tgt:,} targets")

In [ ]:
# Step 4 — Determine shard membership and define manifest splitter

def collect_source_ids(jsonl_path: Path) -> set:
    ids = set()
    with open(str(jsonl_path), encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                row = json.loads(line)
                sid = row.get('source_id')
                if sid:
                    ids.add(str(sid))
    return ids

shard1_sources = collect_source_ids(SHARD1_CACHE)
shard2_sources = collect_source_ids(SHARD2_CACHE)
print(f"Shard 1: {len(shard1_sources)} unique source IDs")
print(f"Shard 2: {len(shard2_sources)} unique source IDs")

overlap = shard1_sources & shard2_sources
if overlap:
    print(f"WARNING: {len(overlap)} source IDs appear in both shards — shard 1 wins for those.")


def _rewrite_path(p: str, local_root: Path) -> str:
    """Rewrite a manifest image path to an absolute local path.

    SSCD/CLIP manifests use relative paths like "source/foo.jpg" or "target/foo.jpg"
    (relative to the CWD where retrieve.py ran). Taking path.name and joining with
    local_root gives the correct absolute path after extraction.
    """
    path = Path(p)
    if path.is_absolute():
        # Unexpected for SSCD/CLIP manifests — return as-is and let ASpanFormer fail
        # with a clear FileNotFoundError rather than silently producing a wrong path.
        return p
    return str(local_root / path.name)


def split_manifest_by_shard(manifest_path: Path, shard_sources: set, out_path: Path) -> int:
    """Write rows whose source_id is in shard_sources to out_path.

    Rewrites source_path and target_path to absolute local paths so
    ASpanFormer reads from the locally-extracted image directories.
    Returns row count.
    """
    rows = []
    with open(str(manifest_path), encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            if str(row.get('source_id', '')) not in shard_sources:
                continue
            if 'source_path' in row:
                row['source_path'] = _rewrite_path(row['source_path'], LOCAL_SOURCE_DIR.resolve())
            if 'target_path' in row:
                row['target_path'] = _rewrite_path(row['target_path'], LOCAL_TARGET_DIR.resolve())
            rows.append(json.dumps(row, ensure_ascii=False))
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text('\n'.join(rows) + ('\n' if rows else ''), encoding='utf-8')
    return len(rows)

# Spot-check: verify a rewritten path actually exists
print("\nPath rewrite spot-check (first 3 SSCD rows):")
with open(str(SSCD_RETRIEVAL), encoding='utf-8') as _f:
    _checked = 0
    for _line in _f:
        _line = _line.strip()
        if not _line:
            continue
        _row = json.loads(_line)
        _src = _rewrite_path(_row['source_path'], LOCAL_SOURCE_DIR.resolve())
        _tgt = _rewrite_path(_row['target_path'], LOCAL_TARGET_DIR.resolve())
        print(f"  src: {_src}  (exists: {Path(_src).exists()})")
        print(f"  tgt: {_tgt}  (exists: {Path(_tgt).exists()})")
        _checked += 1
        if _checked >= 3:
            break

print("\nShard splitter ready.")

## SSCD — Geometry Filter (Shard 1 then Shard 2)

For each shard:
1. Extract the SSCD retrieval rows for that shard's sources → temp manifest
2. Run `geometry_filter_cached.py` with the shard's existing all-manifest as failure cache
3. Write `aspan_all_manifest.jsonl` and `vggt_candidates_manifest.jsonl` to output dir

Expected wall time: **~45–90 min per shard** (ASpanFormer only runs on cache-miss pairs
and pairs that passed in the original DINO run — estimate 10–30% of candidates).

In [ ]:
# SSCD — Shard 1
import importlib
import geometry_filter_cached
importlib.reload(geometry_filter_cached)

SSCD_S1_INPUT  = LOCAL_WORK / 'sscd_shard1_candidates.jsonl'
SSCD_S1_OUTPUT = LOCAL_WORK / 'sscd_shard1_output'

n = split_manifest_by_shard(SSCD_RETRIEVAL, shard1_sources, SSCD_S1_INPUT)
print(f"SSCD shard 1 input: {n} candidate rows")

geometry_filter_cached.main([
    '--input-manifest',   str(SSCD_S1_INPUT),
    '--output-dir',       str(SSCD_S1_OUTPUT),
    '--breakpoint-value', str(BREAKPOINT),
    '--aspanpath',        ASPANPATH,
    '--weights_path',     str(WEIGHTS_PATH),
    '--config_path',      CONFIG_PATH,
    '--cache-manifests',  str(SHARD1_CACHE),
    '--device',           'auto',
    '--progress-every',   '100',
])

print(f"\nSSCD shard 1 complete. Output: {SSCD_S1_OUTPUT}")

In [ ]:
# SSCD — Shard 2
importlib.reload(geometry_filter_cached)

SSCD_S2_INPUT  = LOCAL_WORK / 'sscd_shard2_candidates.jsonl'
SSCD_S2_OUTPUT = LOCAL_WORK / 'sscd_shard2_output'

n = split_manifest_by_shard(SSCD_RETRIEVAL, shard2_sources, SSCD_S2_INPUT)
print(f"SSCD shard 2 input: {n} candidate rows")

geometry_filter_cached.main([
    '--input-manifest',   str(SSCD_S2_INPUT),
    '--output-dir',       str(SSCD_S2_OUTPUT),
    '--breakpoint-value', str(BREAKPOINT),
    '--aspanpath',        ASPANPATH,
    '--weights_path',     str(WEIGHTS_PATH),
    '--config_path',      CONFIG_PATH,
    '--cache-manifests',  str(SHARD2_CACHE),
    '--device',           'auto',
    '--progress-every',   '100',
])

print(f"\nSSCD shard 2 complete. Output: {SSCD_S2_OUTPUT}")

## CLIP — Geometry Filter (Shard 1 then Shard 2)

In [ ]:
# CLIP — Shard 1
importlib.reload(geometry_filter_cached)

CLIP_S1_INPUT  = LOCAL_WORK / 'clip_shard1_candidates.jsonl'
CLIP_S1_OUTPUT = LOCAL_WORK / 'clip_shard1_output'

n = split_manifest_by_shard(CLIP_RETRIEVAL, shard1_sources, CLIP_S1_INPUT)
print(f"CLIP shard 1 input: {n} candidate rows")

geometry_filter_cached.main([
    '--input-manifest',   str(CLIP_S1_INPUT),
    '--output-dir',       str(CLIP_S1_OUTPUT),
    '--breakpoint-value', str(BREAKPOINT),
    '--aspanpath',        ASPANPATH,
    '--weights_path',     str(WEIGHTS_PATH),
    '--config_path',      CONFIG_PATH,
    '--cache-manifests',  str(SHARD1_CACHE),
    '--device',           'auto',
    '--progress-every',   '100',
])

print(f"\nCLIP shard 1 complete. Output: {CLIP_S1_OUTPUT}")

In [ ]:
# CLIP — Shard 2
importlib.reload(geometry_filter_cached)

CLIP_S2_INPUT  = LOCAL_WORK / 'clip_shard2_candidates.jsonl'
CLIP_S2_OUTPUT = LOCAL_WORK / 'clip_shard2_output'

n = split_manifest_by_shard(CLIP_RETRIEVAL, shard2_sources, CLIP_S2_INPUT)
print(f"CLIP shard 2 input: {n} candidate rows")

geometry_filter_cached.main([
    '--input-manifest',   str(CLIP_S2_INPUT),
    '--output-dir',       str(CLIP_S2_OUTPUT),
    '--breakpoint-value', str(BREAKPOINT),
    '--aspanpath',        ASPANPATH,
    '--weights_path',     str(WEIGHTS_PATH),
    '--config_path',      CONFIG_PATH,
    '--cache-manifests',  str(SHARD2_CACHE),
    '--device',           'auto',
    '--progress-every',   '100',
])

print(f"\nCLIP shard 2 complete. Output: {CLIP_S2_OUTPUT}")

## Copy outputs to Drive

In [ ]:
# Copy all 4 output directories (manifests + sidecars) to Drive
import os

output_dirs = [
    (SSCD_S1_OUTPUT, DRIVE_OUT / 'sscd_shard1'),
    (SSCD_S2_OUTPUT, DRIVE_OUT / 'sscd_shard2'),
    (CLIP_S1_OUTPUT, DRIVE_OUT / 'clip_shard1'),
    (CLIP_S2_OUTPUT, DRIVE_OUT / 'clip_shard2'),
]

for src_dir, dst_dir in output_dirs:
    src_dir = Path(str(src_dir))
    dst_dir = Path(str(dst_dir))
    if not src_dir.exists():
        print(f"MISSING: {src_dir.name} — check for errors above")
        continue
    dst_dir.mkdir(parents=True, exist_ok=True)
    # Copy manifests
    for fname in ['aspan_all_manifest.jsonl', 'vggt_candidates_manifest.jsonl']:
        f = src_dir / fname
        if f.exists():
            shutil.copy(str(f), str(dst_dir / fname))
            size_mb = f.stat().st_size / 1e6
            print(f"  Saved: {dst_dir.name}/{fname}  ({size_mb:.1f} MB)")
        else:
            print(f"  MISSING: {dst_dir.name}/{fname}")
    # Copy sidecars directory
    sidecar_src = src_dir / 'aspan_sidecars'
    sidecar_dst = dst_dir / 'aspan_sidecars'
    if sidecar_src.exists():
        if sidecar_dst.exists():
            shutil.rmtree(str(sidecar_dst))
        shutil.copytree(str(sidecar_src), str(sidecar_dst))
        n_npz = len(list(sidecar_dst.glob('*.npz')))
        print(f"  Saved: {dst_dir.name}/aspan_sidecars/  ({n_npz} NPZ files)")
    else:
        print(f"  No sidecars (0 pairs passed threshold) for {dst_dir.name}")

print(f"\nAll outputs at: {DRIVE_OUT}")

## Verify outputs

Prints row counts, cache hit rates, and a sample row for each shard+model combination.

In [ ]:
for model_tag, s1_out, s2_out in [
    ('SSCD', SSCD_S1_OUTPUT, SSCD_S2_OUTPUT),
    ('CLIP', CLIP_S1_OUTPUT, CLIP_S2_OUTPUT),
]:
    for shard_tag, out_dir in [('shard1', s1_out), ('shard2', s2_out)]:
        out_dir = Path(str(out_dir))
        all_path  = out_dir / 'aspan_all_manifest.jsonl'
        pass_path = out_dir / 'vggt_candidates_manifest.jsonl'

        if not all_path.exists():
            print(f"{model_tag} {shard_tag}: all manifest not found\n")
            continue

        all_rows  = [json.loads(l) for l in all_path.read_text().splitlines() if l.strip()]
        pass_rows = [json.loads(l) for l in pass_path.read_text().splitlines() if l.strip()] \
                    if pass_path.exists() else []

        n_total    = len(all_rows)
        n_pass     = len(pass_rows)
        n_cached   = sum(1 for r in all_rows if r.get('from_cache'))
        n_fresh    = n_total - n_cached
        n_except   = sum(1 for r in all_rows if r.get('exception'))
        n_new      = sum(1 for r in all_rows if r.get('new_pair'))
        n_new_pass = sum(1 for r in pass_rows if r.get('new_pair'))

        print(f"── {model_tag} {shard_tag} {'─' * 35}")
        print(f"  All pairs     : {n_total:,}")
        print(f"  Passed filter : {n_pass:,}  ({100*n_pass/n_total:.1f}%)")
        print(f"  Cache hits    : {n_cached:,}  ({100*n_cached/n_total:.1f}% skipped ASpanFormer)")
        print(f"  Fresh runs    : {n_fresh:,}")
        print(f"  Exceptions    : {n_except:,}")
        print(f"  New pairs     : {n_new:,}  ({100*n_new/n_total:.1f}% not seen in DINO pipeline)")
        if n_new > 0:
            pct_new_pass = 100 * n_new_pass / n_new
            print(f"    └ of new pairs, passed filter: {n_new_pass}/{n_new}  ({pct_new_pass:.1f}%)")

        sidecar_dir = out_dir / 'aspan_sidecars'
        n_npz = len(list(sidecar_dir.glob('*.npz'))) if sidecar_dir.exists() else 0
        print(f"  Sidecar NPZs  : {n_npz:,}  (should equal passed count)")

        if pass_rows:
            r0 = pass_rows[0]
            print(f"  Sample pass   : {r0.get('source_id')} → {r0.get('target_id')}  "
                  f"kp={r0.get('filtered_keypoint_count')}  "
                  f"new={r0.get('new_pair')}  sidecar={'sidecar_path' in r0}")
        print()